# অলীকবচন v11 — Merged Judge System (Team Outliers)

**Union of the team's two independently-built pipelines** (Shahriar's router/math/trap system,
scored 0.755; Ahan's logprob/ladder system, scored 0.803).

## Approach
1. **Three-route routing**: `grounded` (context present) / `closedbook` (`[NULL]`) / `math` (numeric reasoning).
2. **Dual scoring per row** — every non-math row gets BOTH:
   - `P(faithful)` from first-token **Yes/No logprobs** (greedy, continuous, cheap), and
   - **CoT self-consistency**: short English reasoning → `Verdict: Yes/No`, sampled k times.
   Final score = mean of the two. Continuous scores → far better threshold calibration than vote fractions.
3. **Math route**: the judge *solves the problem itself*, then compares answers (k=3 votes).
4. **RAG**: BM25 (pure-python, offline) over any attached Bengali wiki/GK corpus; evidence is
   injected into closed-book prompts. Auto-skips if nothing attached.
5. **Trap-aware grounded prompt**: verbatim context overlap does not equal faithful; the answer
   must match the question's *type* (year vs name vs number).
6. **Calibration** on the labelled sample split (minus few-shot exemplars), one threshold per route,
   optimized for `METRIC` (default **macro-F1**, per the Kaggle Evaluation tab); both metrics printed.
7. Engine ladder: Qwen2.5-32B-AWQ → 14B-AWQ (vLLM) → bnb-4bit fallback. Disk cache lets an
   interrupted interactive session resume.

## Attachments
competition data · (optional but recommended) a public Bengali wiki/GK dataset, e.g.
`shazol/bangla-wikipedia` — declare it on the Discussion tab · (optional) vLLM wheels for offline Phase-2.

## Compliance
Open weights only, local inference, T4×2, <9h, <50GB, schema-robust loading, no test-set training,
external data public+declared+cited. Reproduces Phase-1 predictions end-to-end (Phase-2 ready).


In [ ]:
# ---- engine install: MUST run before torch is imported anywhere in this kernel ----
import importlib.util, subprocess, glob as _g, os
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

if importlib.util.find_spec("vllm") is None:
    whls = sorted(_g.glob("/kaggle/input/**/vllm*.whl", recursive=True))
    if whls:
        subprocess.run(["pip", "install", "-q", "--no-index",
                        "--find-links", whls[0].rsplit("/", 1)[0], "vllm"], check=False)
    else:
        print("installing vllm (internet ON needed; failure is OK -> bnb fallback)")
        subprocess.run(["pip", "install", "-q", "vllm"], check=False)
if importlib.util.find_spec("bitsandbytes") is None:
    subprocess.run(["pip", "install", "-q", "-U", "bitsandbytes", "accelerate"], check=False)
print("engine install cell done")


In [ ]:
import os, re, gc, json, math, time, glob, hashlib, random
import numpy as np, pandas as pd

SEED = 42
random.seed(SEED); np.random.seed(SEED)

# ---------------- toggles ----------------
METRIC        = "macro"    # "macro" (Kaggle Evaluation tab) | "f1_0" (rulebook wording)
K_GROUNDED    = 3          # CoT samples on grounded rows
K_CLOSEDBOOK  = 5          # CoT samples on closed-book rows
K_MATH        = 3          # solve-and-compare samples
COT_TOKENS    = 260        # English reasoning is short; verdict never truncates
MATH_TOKENS   = 560
CTX_CLIP, Q_CLIP, R_CLIP = 2000, 500, 700
EVIDENCE_K, EVIDENCE_CHARS = 4, 450
JUDGE_CHUNK   = 256
RAG_MAX_PASSAGES = 400_000

# ladder: (hf_id_or_localdir, tensor_parallel, max_model_len)
JUDGE_LADDER = [
    ("Qwen/Qwen2.5-32B-Instruct-AWQ", 2, 3072),
    ("Qwen/Qwen2.5-32B-Instruct-AWQ", 2, 2560),
    ("Qwen/Qwen2.5-14B-Instruct-AWQ", 2, 4096),
    ("Qwen/Qwen2.5-14B-Instruct-AWQ", 1, 3072),
]
JUDGE_FALLBACK_BNB = "Qwen/Qwen2.5-14B-Instruct"

# NVRTC / JIT-fusion workaround (proven fix from the team's v9 crash)
os.environ.setdefault("PYTORCH_NVFUSER_DISABLE", "fallback")
try:
    import torch
    torch.manual_seed(SEED)
    for _fn, _arg in [("_jit_set_texpr_fuser_enabled", False),
                      ("_jit_override_can_fuse_on_gpu", False),
                      ("_jit_override_can_fuse_on_cpu", False),
                      ("_jit_set_nvfuser_enabled", False)]:
        try: getattr(torch._C, _fn)(_arg)
        except Exception: pass
    N_GPU = max(torch.cuda.device_count(), 1)
except Exception:
    torch, N_GPU = None, 0

def find_test_csv():
    cands = [p for p in sorted(glob.glob("/kaggle/input/**/*.csv", recursive=True))
             if os.path.basename(p).lower().replace(" ", "").startswith("test")]
    assert cands, "test csv not found under /kaggle/input"
    return cands[0]

def find_labeled_sample():
    for p in sorted(glob.glob("/kaggle/input/**/*", recursive=True)):
        b = os.path.basename(p).lower()
        if b.replace(" ", "").startswith("train") and b.endswith(".csv"):
            return p
    for p in sorted(glob.glob("/kaggle/input/**/*.json", recursive=True)):
        if "sample" in os.path.basename(p).lower() and "submission" not in p.lower():
            return p
    return None

def find_local_model():
    for cfg in sorted(glob.glob("/kaggle/input/**/config.json", recursive=True)):
        try: c = json.load(open(cfg))
        except Exception: continue
        if "qwen2" in (str(c.get("architectures", "")) + str(c.get("model_type", ""))).lower():
            return os.path.dirname(cfg)
    return None

def find_wiki():
    hits = sorted(glob.glob("/kaggle/input/**/*passages*.parquet", recursive=True))
    if hits: return hits[0]
    bad = ("testset", "test set", "datasetsamples", "dataset samples", "samplesubmission",
           "sample submission", "synthetic_train", "submission")
    cand = []
    for ext in ("*.parquet", "*.csv", "*.json", "*.jsonl", "*.txt", "*.tsv"):
        for p in glob.glob(f"/kaggle/input/**/{ext}", recursive=True):
            pl = p.lower()
            if any(b in pl for b in bad): continue
            if any(k in pl for k in ("wiki", "passage", "corpus", "bangla", "bengali", "article"))                     and os.path.getsize(p) > 1_000_000:
                cand.append(p)
    return sorted(cand, key=os.path.getsize)[-1] if cand else None

TEST_CSV, SAMPLE_LB = find_test_csv(), find_labeled_sample()
LOCAL_MODEL, WIKI_SRC = find_local_model(), find_wiki()
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
print("test:", TEST_CSV, "\nlabeled:", SAMPLE_LB, "\nlocal model:", LOCAL_MODEL,
      "\nwiki:", WIKI_SRC or "NONE (closed-book runs knowledge-only)", "\nGPUs:", N_GPU)


In [ ]:
# ---------------- load & route (never hardcode row count/order/ids) ----------------
BN2EN = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")
MATH_KW = re.compile(
    r"শতকরা|সম্ভাবনা|গুণিতক|মৌলিক|ধারাটি|সমষ্টি|যোগফল|বর্গ(?:মূল|ইঞ্চি|ফুট)?|"
    r"ক্ষতি হ|লাভ হ|বয়স|কত ?দিনে|গড়|অনুপাত|ভগ্নাংশ|সুদ|আসল|"
    r"[0-9০-৯]\s*[+\-*/=]|√|x\s*[*+=]|\*\*|%")

def has_context(v):
    if v is None or (isinstance(v, float) and np.isnan(v)): return False
    return str(v).strip() not in ("", "[NULL]", "NULL", "nan")

def route_row(ctx, q):
    if MATH_KW.search(str(q)): return "math"
    return "grounded" if has_context(ctx) else "closedbook"

def load_frame(path):
    df = pd.read_json(path) if path.endswith(".json") else pd.read_csv(path)
    low = {c.lower().strip(): c for c in df.columns}
    cols = dict(ctx=low.get("context"), q=low.get("prompt_bn") or low.get("prompt"),
                r=low.get("response_bn") or low.get("response"),
                id=low.get("id"), y=low.get("label"))
    assert cols["q"] and cols["r"], f"unexpected schema: {list(df.columns)}"
    rts = np.array([route_row(df[cols["ctx"]].iloc[i] if cols["ctx"] else None,
                              df[cols["q"]].iloc[i]) for i in range(len(df))])
    return df, cols, rts

test, tcols, test_routes = load_frame(TEST_CSV)
ids = test[tcols["id"]].values if tcols["id"] else np.arange(1, len(test) + 1)
print(len(test), "test rows | routes:", pd.Series(test_routes).value_counts().to_dict())

train, rcols, train_routes = (None, None, None)
if SAMPLE_LB:
    train, rcols, train_routes = load_frame(SAMPLE_LB)
    if rcols["y"] is None: train = None
    else: print(len(train), "labeled rows | routes:", pd.Series(train_routes).value_counts().to_dict())


In [ ]:
# ---------------- offline BM25 retrieval ----------------
TOK_RE = re.compile(r"[\wঀ-৿]+")
STOP = set("কী কি কে কার কাকে কোন কোথায় কবে হয় ছিল ছিলেন এর একটি ও এবং থেকে সালে করে করা হয়েছিল যে এই".split())

def tokenize(s):
    return [t for t in TOK_RE.findall(str(s).translate(BN2EN).lower()) if t not in STOP and len(t) > 1]

class BM25:
    def __init__(self, docs, k1=1.5, b=0.75):
        self.docs, self.k1, self.b = docs, k1, b
        self.post, self.dl = {}, np.zeros(len(docs), dtype=np.float32)
        for i, d in enumerate(docs):
            toks = tokenize(d); self.dl[i] = len(toks) or 1
            tf = {}
            for t in toks: tf[t] = tf.get(t, 0) + 1
            for t, f in tf.items(): self.post.setdefault(t, []).append((i, f))
        self.avgdl = float(self.dl.mean()) if len(docs) else 1.0
        self.N = len(docs)
    def top(self, query, k=4):
        sc = {}
        for t in set(tokenize(query)):
            pl = self.post.get(t)
            if not pl: continue
            idf = np.log(1 + (self.N - len(pl) + 0.5) / (len(pl) + 0.5))
            for i, f in pl:
                den = f + self.k1 * (1 - self.b + self.b * self.dl[i] / self.avgdl)
                sc[i] = sc.get(i, 0.0) + idf * f * (self.k1 + 1) / den
        return [self.docs[i] for i, _ in sorted(sc.items(), key=lambda x: -x[1])[:k]]

def load_corpus(path):
    out = []
    def add(text, title=""):
        text = str(text)
        for s0 in range(0, min(len(text), 1800), 900):
            ch = text[s0:s0 + 900].strip()
            if len(ch) >= 80: out.append((title + ": " if title else "") + ch)
    if path.endswith(".txt"):
        for block in open(path, encoding="utf-8", errors="ignore").read().split("\n\n"):
            add(block)
            if len(out) >= RAG_MAX_PASSAGES: return out
        return out
    if path.endswith(".parquet"): d = pd.read_parquet(path)
    elif path.endswith(".tsv"):   d = pd.read_csv(path, sep="\t", on_bad_lines="skip")
    elif path.endswith(".csv"):   d = pd.read_csv(path, on_bad_lines="skip")
    else:
        try: d = pd.read_json(path, lines=path.endswith(".jsonl"))
        except ValueError: d = pd.read_json(path, lines=True)
    obj = [c for c in d.columns if d[c].dtype == object]
    tcol = max(obj, key=lambda c: d[c].astype(str).str.len().mean()) if obj else d.columns[-1]
    ttl = next((c for c in obj if c != tcol and "title" in str(c).lower()), None)
    for _, row in d.iterrows():
        add(row[tcol], str(row[ttl]) if ttl else "")
        if len(out) >= RAG_MAX_PASSAGES: break
    return out

bm25 = None
if WIKI_SRC:
    t0 = time.time()
    _p = load_corpus(WIKI_SRC)
    bm25 = BM25(_p)
    gc.collect()
    print(f"BM25 over {len(_p)} passages in {time.time()-t0:.0f}s")


In [ ]:
# ---------------- JUDGE ENGINE: vLLM ladder + logprob + sampling + cache ----------------
from transformers import AutoTokenizer
ENGINE, LLM_OBJ, MODEL_SRC, MODEL_MAXLEN, tok = None, None, None, 3072, None

def _try_vllm():
    global ENGINE, LLM_OBJ, MODEL_SRC, MODEL_MAXLEN
    try:
        import vllm  # noqa
    except ImportError:
        print("vllm unavailable"); return False
    from vllm import LLM
    ladder = ([(LOCAL_MODEL, min(N_GPU, 2), 3072)] if LOCAL_MODEL else []) + JUDGE_LADDER
    for src, tp, ml in ladder:
        if tp > max(N_GPU, 1): continue
        try:
            print(f">>> trying {src} (tp={tp}, max_len={ml})")
            LLM_OBJ = LLM(model=src, quantization="awq" if "awq" in src.lower() else None,
                          dtype="half", tensor_parallel_size=tp, max_model_len=ml,
                          gpu_memory_utilization=0.90, enforce_eager=True,
                          swap_space=2, disable_log_stats=True)
            ENGINE, MODEL_SRC, MODEL_MAXLEN = "vllm", src, ml
            return True
        except Exception as e:
            print(f"    failed: {type(e).__name__}: {str(e)[:160]}")
            LLM_OBJ = None; gc.collect()
            if torch is not None: torch.cuda.empty_cache()
    return False

def _load_bnb():
    global ENGINE, LLM_OBJ, MODEL_SRC, MODEL_MAXLEN
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig
    MODEL_SRC = JUDGE_FALLBACK_BNB
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                             bnb_4bit_quant_type="nf4")
    LLM_OBJ = AutoModelForCausalLM.from_pretrained(
        MODEL_SRC, device_map="auto", quantization_config=bnb, low_cpu_mem_usage=True)
    LLM_OBJ.eval()
    ENGINE, MODEL_MAXLEN = "bnb", 3072

if not _try_vllm():
    _load_bnb()
tok = AutoTokenizer.from_pretrained(MODEL_SRC)
if tok.pad_token is None: tok.pad_token = tok.eos_token
tok.padding_side = "left"
print("engine:", ENGINE, "| model:", MODEL_SRC, "| max_len:", MODEL_MAXLEN)

SYS = ("You are a meticulous bilingual (Bengali/English) fact-checker. You judge whether a "
       "Bengali answer is factually correct and, when a passage is given, supported by it.")

def chatfmt(user_msg):
    return tok.apply_chat_template([{"role": "system", "content": SYS},
                                    {"role": "user", "content": user_msg}],
                                   tokenize=False, add_generation_prompt=True)

def _fit(txt, out_tokens):
    ids = tok.encode(txt)
    lim = MODEL_MAXLEN - out_tokens - 8
    return tok.decode(ids[:lim]) if len(ids) > lim else txt

YES_SET, NO_SET = {"yes", "y", "true"}, {"no", "n", "false"}

def logprob_pfaithful(user_msgs):
    """P(faithful) from first-token Yes/No logprobs (greedy, 1 token)."""
    prompts = [_fit(chatfmt(u), 4) for u in user_msgs]
    if ENGINE == "vllm":
        from vllm import SamplingParams
        outs = LLM_OBJ.generate(prompts, SamplingParams(temperature=0.0, max_tokens=1, logprobs=20))
        res = []
        for o in outs:
            py = pn = 0.0
            lp = o.outputs[0].logprobs
            if lp:
                for tid, l in lp[0].items():
                    s = (l.decoded_token if l.decoded_token is not None else tok.decode([tid])).strip().lower()
                    if s in YES_SET: py += math.exp(l.logprob)
                    elif s in NO_SET: pn += math.exp(l.logprob)
            res.append(py / (py + pn) if (py + pn) > 1e-9 else 0.5)
        return res
    res = []
    yes_ids = {tok.encode(w, add_special_tokens=False)[0] for w in ["Yes", "yes", " Yes"]}
    no_ids  = {tok.encode(w, add_special_tokens=False)[0] for w in ["No", "no", " No"]}
    for s0 in range(0, len(prompts), 8):
        enc = tok(prompts[s0:s0+8], return_tensors="pt", padding=True,
                  truncation=True, max_length=MODEL_MAXLEN - 8).to(LLM_OBJ.device)
        with torch.no_grad():
            logits = LLM_OBJ(**enc).logits
        last = enc["attention_mask"].sum(1) - 1
        for bi in range(logits.shape[0]):
            pr = torch.softmax(logits[bi, last[bi]].float(), -1)
            py = float(sum(pr[i] for i in yes_ids)); pn = float(sum(pr[i] for i in no_ids))
            res.append(py / (py + pn) if (py + pn) > 1e-9 else 0.5)
    return res

def sample_texts(user_msgs, max_tokens, k):
    """k sampled completions per message -> list[list[str]]."""
    prompts = [_fit(chatfmt(u), max_tokens) for u in user_msgs]
    if ENGINE == "vllm":
        from vllm import SamplingParams
        sp = SamplingParams(temperature=0.7, top_p=0.9, max_tokens=max_tokens, n=k)
        return [[c.text for c in o.outputs] for o in LLM_OBJ.generate(prompts, sp)]
    res = []
    for s0 in range(0, len(prompts), 4):
        chunk = prompts[s0:s0+4]
        enc = tok(chunk, return_tensors="pt", padding=True, truncation=True,
                  max_length=MODEL_MAXLEN - max_tokens).to(LLM_OBJ.device)
        with torch.no_grad():
            g = LLM_OBJ.generate(**enc, max_new_tokens=max_tokens, do_sample=True,
                                 temperature=0.7, top_p=0.9, num_return_sequences=k,
                                 pad_token_id=tok.eos_token_id)
        g = g[:, enc["input_ids"].shape[1]:]
        texts = tok.batch_decode(g, skip_special_tokens=True)
        for i in range(len(chunk)):
            res.append(texts[i*k:(i+1)*k])
    return res

# disk cache: interactive session restarts resume instead of recomputing
CACHE_PATH = os.path.join(WORK, f"judge_cache_{(MODEL_SRC or 'x').replace('/','_')}.json")
try: _cache = json.load(open(CACHE_PATH))
except Exception: _cache = {}

def _chunked(fn, msgs, kind, **kw):
    keys = [kind + hashlib.sha1(m.encode()).hexdigest() for m in msgs]
    out = [_cache.get(k) for k in keys]
    pend = [i for i, v in enumerate(out) if v is None]
    for s0 in range(0, len(pend), JUDGE_CHUNK):
        blk = pend[s0:s0 + JUDGE_CHUNK]
        for i, r in zip(blk, fn([msgs[i] for i in blk], **kw)):
            out[i] = r; _cache[keys[i]] = r
        json.dump(_cache, open(CACHE_PATH, "w"))
        print(f"    {kind} {min(s0+JUDGE_CHUNK, len(pend))}/{len(pend)}", flush=True)
    return out


In [ ]:
# ---------------- prompts (English scaffolding, Bengali content) ----------------
def _clip(s, n):
    s = str(s) if s is not None else ""
    return s if len(s) <= n else s[:n] + "…"

# few-shot exemplars: shortest 2 faithful + 2 hallucinated closed-book rows (excluded from calibration)
FEWSHOT_IDX, FEWSHOT = [], ""
if train is not None:
    _nc = train[[not has_context(train[rcols["ctx"]].iloc[i]) if rcols["ctx"] else True
                 for i in range(len(train))]].copy()
    if len(_nc) >= 4:
        _nc["tl"] = _nc[rcols["q"]].astype(str).str.len() + _nc[rcols["r"]].astype(str).str.len()
        FEWSHOT_IDX = (list(_nc[_nc[rcols["y"]] == 1].sort_values("tl").index[:2]) +
                       list(_nc[_nc[rcols["y"]] == 0].sort_values("tl").index[:2]))
        FEWSHOT = "\n\n".join(
            f"Question: {_clip(train.at[i, rcols['q']], 200)}\n"
            f"Proposed answer: {_clip(train.at[i, rcols['r']], 200)}\n"
            f"Verdict: {'Yes' if int(train.at[i, rcols['y']]) == 1 else 'No'}"
            for i in FEWSHOT_IDX)
print("few-shot exemplar rows (excluded from calibration):", FEWSHOT_IDX)

TRAP_RULES = (
    "Rules:\n"
    "- The response is faithful ONLY if it answers exactly what the question asks AND is factually correct.\n"
    "- Copying words from the passage does NOT make it faithful: if the question asks for a year, a name "
    "is wrong; if the question's premise is not supported by the passage, be skeptical.\n"
    "- Partially wrong, wrong entity/date/number, or fabricated details => hallucinated.\n")

def p_grounded(ctx, q, r, want):
    base = (f"Passage (Bengali):\n{_clip(ctx, CTX_CLIP)}\n\n"
            f"Question (Bengali): {_clip(q, Q_CLIP)}\nResponse (Bengali): {_clip(r, R_CLIP)}\n\n" + TRAP_RULES)
    if want == "lp":
        return base + "Is the response faithful? Answer with exactly one word: Yes or No."
    return base + ("Reason in 1-3 short sentences, then on a new line output exactly "
                   "\"Verdict: Yes\" (faithful) or \"Verdict: No\" (hallucinated).")

def p_closedbook(q, r, ev, want):
    head = ("Solved examples:\n\n" + FEWSHOT + "\n\n") if FEWSHOT else ""
    evb = ("Evidence passages (Bengali Wikipedia):\n" +
           "\n".join("- " + _clip(e, EVIDENCE_CHARS) for e in ev) + "\n\n") if ev else ""
    base = (head + evb + "Judge using your own knowledge of Bengali/Bangladesh facts"
            + (" and the evidence above" if ev else "") +
            f".\nQuestion (Bengali): {_clip(q, Q_CLIP)}\nProposed answer (Bengali): {_clip(r, R_CLIP)}\n")
    if want == "lp":
        return base + "Is the proposed answer factually correct? Answer with exactly one word: Yes or No.\nVerdict:"
    return base + ("Reason in 1-3 short sentences, then on a new line output exactly "
                   "\"Verdict: Yes\" (correct) or \"Verdict: No\" (wrong/fabricated).")

def p_math(q, r):
    return ("Solve this Bengali math problem step by step yourself. Then compare YOUR answer with the "
            "proposed answer (treat equal values written differently as a match, e.g. ১/২ = 0.5).\n"
            f"Problem (Bengali): {_clip(q, 900)}\nProposed answer (Bengali): {_clip(r, R_CLIP)}\n"
            "End with exactly \"Verdict: Yes\" if they match, or \"Verdict: No\" if they do not.")

VERDICT_EN = re.compile(r"verdict\s*[:：]?\s*(yes|no)", re.I)
VERDICT_BN = re.compile(r"রায়\s*[:ঃ]?\s*([01])")

def parse_verdict(text):
    t = str(text)
    m = VERDICT_EN.findall(t)
    if m: return 1 if m[-1].lower() == "yes" else 0
    m = VERDICT_BN.findall(t.translate(BN2EN))
    if m: return int(m[-1])
    tl = t.strip().lower()
    if tl.startswith("yes"): return 1
    if tl.startswith("no"): return 0
    return None   # abstain — never silently counted as a verdict

def votes_score(comps):
    vs = [v for v in (parse_verdict(c) for c in comps) if v is not None]
    return (float(np.mean(vs)) if vs else None)


In [ ]:
# ---------------- scoring: dual signal per row ----------------
def score_frame(df, cols, rts, tag):
    n = len(df)
    get = lambda c, i: df[cols[c]].iloc[i] if cols[c] else None
    idx = {"grounded": [], "closedbook": [], "math": []}
    for i in range(n): idx[rts[i]].append(i)
    print(f"[{tag}] routes:", {k: len(v) for k, v in idx.items()})
    scores, t0 = np.full(n, 0.5, dtype=np.float32), time.time()
    ev_cache = {}
    def evidence(i):
        if bm25 is None: return []
        if i not in ev_cache:
            ev_cache[i] = bm25.top(str(get("q", i)) + " " + str(get("r", i)), k=EVIDENCE_K)
        return ev_cache[i]

    # grounded: logprob + CoT(k)
    g = idx["grounded"]
    if g:
        lp = _chunked(logprob_pfaithful, [p_grounded(get("ctx", i), get("q", i), get("r", i), "lp") for i in g], f"{tag}-g-lp")
        ct = _chunked(sample_texts, [p_grounded(get("ctx", i), get("q", i), get("r", i), "cot") for i in g],
                      f"{tag}-g-cot", max_tokens=COT_TOKENS, k=K_GROUNDED)
        for j, i in enumerate(g):
            v = votes_score(ct[j])
            scores[i] = (lp[j] + v) / 2 if v is not None else lp[j]

    # closed-book: evidence + logprob + CoT(k)
    c = idx["closedbook"]
    if c:
        lp = _chunked(logprob_pfaithful, [p_closedbook(get("q", i), get("r", i), evidence(i), "lp") for i in c], f"{tag}-c-lp")
        ct = _chunked(sample_texts, [p_closedbook(get("q", i), get("r", i), evidence(i), "cot") for i in c],
                      f"{tag}-c-cot", max_tokens=COT_TOKENS, k=K_CLOSEDBOOK)
        for j, i in enumerate(c):
            v = votes_score(ct[j])
            scores[i] = (lp[j] + v) / 2 if v is not None else lp[j]

    # math: solve-and-compare votes
    m = idx["math"]
    if m:
        ct = _chunked(sample_texts, [p_math(get("q", i), get("r", i)) for i in m],
                      f"{tag}-math", max_tokens=MATH_TOKENS, k=K_MATH)
        for j, i in enumerate(m):
            v = votes_score(ct[j])
            scores[i] = v if v is not None else 0.5
    print(f"[{tag}] scored {n} rows in {(time.time()-t0)/60:.1f} min")
    return scores

test_scores = score_frame(test, tcols, test_routes, "test")


In [ ]:
# ---------------- calibration (both metrics printed; METRIC optimized) ----------------
def f1_class(y, p, cls):
    y, p = np.asarray(y), np.asarray(p)
    tp = ((p == cls) & (y == cls)).sum(); fp = ((p == cls) & (y != cls)).sum()
    fn = ((p != cls) & (y == cls)).sum()
    return 2*tp / (2*tp + fp + fn) if (2*tp + fp + fn) else 0.0

f1_0 = lambda y, p: f1_class(y, p, 0)
f1_macro = lambda y, p: (f1_class(y, p, 0) + f1_class(y, p, 1)) / 2
metric_fn = f1_macro if METRIC == "macro" else f1_0

def best_threshold(s, y):
    best = (0.5, -1)
    for t in np.linspace(0.02, 0.98, 97):
        v = metric_fn(y, (np.asarray(s) >= t).astype(int))
        if v > best[1]: best = (float(t), v)
    return best

THRESHOLDS = {"grounded": 0.5, "closedbook": 0.5, "math": 0.5}
if train is not None:
    tr_scores = score_frame(train, rcols, train_routes, "calib")
    y = train[rcols["y"]].values.astype(int)
    keep = np.array([i not in set(FEWSHOT_IDX) for i in train.index])
    for rt in THRESHOLDS:
        m = (train_routes == rt) & keep
        if m.sum() >= 10:
            t, v = best_threshold(tr_scores[m], y[m])
            THRESHOLDS[rt] = t
            print(f"{rt:<10} n={m.sum():>3} best_t={t:.2f} {METRIC}={v:.3f}")
    pred = np.array([int(tr_scores[i] >= THRESHOLDS[train_routes[i]]) for i in range(len(train))])
    print(f"sample-split overall: macro={f1_macro(y[keep], pred[keep]):.4f}  "
          f"F1(0)={f1_0(y[keep], pred[keep]):.4f}  (optimized: {METRIC})")
    pd.DataFrame({"route": train_routes, "score": tr_scores, "label": y}).to_csv(
        os.path.join(WORK, "calib_scores.csv"), index=False)
else:
    print("no labeled sample -> default thresholds", THRESHOLDS)


In [ ]:
# ---------------- submission ----------------
labels = np.array([int(test_scores[i] >= THRESHOLDS[test_routes[i]]) for i in range(len(test))])
sub = pd.DataFrame({"id": ids, "label": labels})
assert list(sub.columns) == ["id", "label"] and len(sub) == len(test)
assert sub["label"].isin([0, 1]).all() and sub["id"].notna().all()
sub.to_csv(os.path.join(WORK, "submission.csv"), index=False)
pd.DataFrame({"id": ids, "route": test_routes, "score": test_scores, "label": labels}).to_csv(
    os.path.join(WORK, "test_scores.csv"), index=False)
print("submission:", sub.label.value_counts().to_dict())
for rt in ("grounded", "closedbook", "math"):
    m = test_routes == rt
    print(f"  {rt:<10} n={m.sum():>4}  mean(label)={labels[m].mean():.3f}")
print("Sanity: labelled set is ~55% faithful; if a route collapsed to one class, "
      "re-check the engine line and METRIC before submitting.")


## Runtime & reproduction
- T4×2, vLLM, 32B: logprob passes are prefill-bound (~fast); CoT k=3/5 with 260-token English
  reasoning ≈ 3–5 h total for 2.5k test + 299 calibration rows. Ladder degrades to 14B if 32B
  cannot initialize; bnb fallback is last resort.
- Disk cache (`judge_cache_*.json`) makes interrupted interactive sessions resumable.
- Set `METRIC` after the all-zeros diagnostic; both metrics are printed either way.
- External data must be declared on the Discussion tab: the wiki/GK corpus you attach (+ citation).
